# Deploy Gemma 4 on AMD MI300X/48GB GPU via vLLM

This notebook installs vLLM with ROCm support to run Gemma 4 models efficiently on AMD hardware.
It also sets up Cloudflare Tunnel (or SSH tunnel) to expose the API publicly so your Fuenzer Sports backend can reach it.

In [ ]:
# Upgrade pip, install vLLM, and upgrade transformers for Gemma 4 support
%pip install --upgrade pip
%pip install --upgrade vllm transformers

### Optional: Hugging Face Authentication
Gemma models may require accepting licensing terms on Hugging Face. If you get a `GatedRepoError` or Access Denied, set your `HF_TOKEN` below.

In [ ]:
import os
# Replace with your Hugging Face Read Token if needed
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"

In [ ]:
# OR This
%env HF_TOKEN=your_hf_token

### Step 1: Install Cloudflared Binary
Since `cloudflared` is a Go binary and not on PyPI, we download and install it manually.

In [ ]:
# Download and install cloudflared binary with SSL bypass for AMD Cloud environment
!curl -kL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared || \
 wget -q --no-check-certificate https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

### Step 2: Start vLLM Server
We use Python's `subprocess` to spawn vLLM in the background, since Jupyter doesn't natively support backgrounding shell processes (`&` or `nohup`).

**Note on Models:**
- **`google/gemma-4-12B-it` (Default):** Fits perfectly in a single 48GB GPU (uses ~24GB VRAM).
- **`RedHatAI/gemma-4-31B-it-FP8-dynamic` (Alternative):** Fits ~31GB VRAM, uses quantized 8-bit model.

In [ ]:
import subprocess
import sys

# Choose model:
MODEL = "google/gemma-4-12B-it" # Fits in 48GB VRAM (Standard BF16)
# MODEL = "RedHatAI/gemma-4-31B-it-FP8-dynamic" # Alternative: 31B FP8 Quantized

proc = subprocess.Popen(
    [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL,
        "--port", "8000",
        "--max-model-len", "8192",
        "--enforce-eager"
    ],
    stdout=open("vllm.log", "w"),
    stderr=subprocess.STDOUT
)
print(f"✅ vLLM server started with PID: {proc.pid} using model {MODEL}")
print("Monitor logs with: !tail -f vllm.log")

### Step 3: Check Logs
Wait until the server says `Uvicorn running on http://0.0.0.0:8000` before running the tunnel cell.

In [ ]:
!tail -n 50 vllm.log

### Step 4: Start Tunnel
This will output a public URL. Copy this URL and set it as `LOCAL_MODEL_BASE_URL` in your backend's `.env` file (append `/v1` to it, e.g. `https://xxx.trycloudflare.com/v1`).

#### Option A: Cloudflare Tunnel

In [ ]:
!cloudflared tunnel --url http://localhost:8000

#### Option B: Fallback SSH Tunnel (Run this if Cloudflare fails to connect)

In [ ]:
import subprocess
import time

proc = subprocess.Popen(
    ["ssh", "-o", "StrictHostKeyChecking=no", "-R", "80:localhost:8000", "nokey@localhost.run"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
time.sleep(3)

# Read output to get the public URL
for line in proc.stdout:
    if ".localhost.run" in line:
        print(f"🔗 Public URL (SSH Fallback): {line.strip()}")
        break